# Results Overview: Model Predictions

Visualization of trained models with:
- **Learning Rate:** 0.0001
- **EMA Alpha:** 0.9
- **Image Size:** 512
- **Positive Weights:** 3, 5, 7, 9

For each model, displaying 10 validation samples with raw image, model prediction, ground truth mask, and overlay.


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
from torchvision import models

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import json
import os
import yaml
from pathlib import Path

%matplotlib inline

print("Libraries imported successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Libraries imported successfully!
PyTorch: 2.10.0+cpu
CUDA available: False


In [8]:
# Copy the GuidedBoxModel class from the training notebook
class GuidedBoxModel(nn.Module):
    def __init__(self, config):
        super(GuidedBoxModel, self).__init__()
        
        # Load ResNet50 and extract only convolutional feature layers
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )
        
        self.teacher = nn.Sequential(
            nn.Conv2d(2048, 1024, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(1024, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, config['num_classes'], kernel_size=1),
        )
        self.student = nn.Sequential(
            nn.Conv2d(2048, 1024, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(1024, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, config['num_classes'], kernel_size=1),
        )
        self.alpha = config['ema_alpha']

    def forward(self, images, positive_weight=None, boxes=None, masks=None):
        features = self.backbone(images)
        teacher_output = self.teacher(features)
        student_output = self.student(features)
        
        teacher_output = F.interpolate(teacher_output, size=images.shape[2:], mode='bilinear', align_corners=False)
        student_output = F.interpolate(student_output, size=images.shape[2:], mode='bilinear', align_corners=False)
        
        if self.training:
            loss = self.compute_loss(teacher_output, student_output, boxes=boxes, masks=masks, positive_weight=positive_weight)
            return loss
        else:
            return student_output

    def compute_loss(self, teacher_output, student_output, boxes, masks, positive_weight):
        conf_scores = self.compute_confidence_scores(teacher_output, student_output)
        mask_loss = self.robust_pseudo_mask_loss(student_output, masks, conf_scores, positive_weight=positive_weight)
        box_loss = F.mse_loss(teacher_output, student_output)
        return box_loss + mask_loss

    def robust_pseudo_mask_loss(self, preds, pseudo_masks, conf_scores, positive_weight=5):
        pixel_loss = F.binary_cross_entropy_with_logits(
            preds, pseudo_masks,
            pos_weight=torch.tensor(float(positive_weight), device=preds.device),
        )
        affinity_loss = self.enhanced_mask_affinity_loss(preds, pseudo_masks)
        return torch.mean(conf_scores * (0.4 * pixel_loss + 0.1 * affinity_loss))

    def enhanced_mask_affinity_loss(self, preds, pseudo_masks):
        eps = 1e-7
        affinity_loss = 0.0
        for i in range(preds.size(0)):
            mask_pred = torch.sigmoid(preds[i])
            mask_pred = torch.clamp(mask_pred, eps, 1 - eps)
            
            neighbors = F.max_pool2d(
                mask_pred.unsqueeze(0), kernel_size=3, stride=1, padding=1
            ).squeeze(0) > 0.45
            
            fg_loss = 0.0
            bg_loss = 0.0
            if neighbors.sum() > 0:
                fg_loss = -torch.mean(torch.log(mask_pred[neighbors]))
            if (~neighbors).sum() > 0:
                bg_loss = -torch.mean(torch.log(1 - mask_pred[~neighbors]))
            
            affinity_loss += fg_loss + bg_loss
        return affinity_loss / preds.size(0)

    def compute_confidence_scores(self, teacher_output, student_output):
        return torch.sigmoid(F.cosine_similarity(teacher_output, student_output))
    
    def update_teacher(self):
        for t_param, s_param in zip(self.teacher.parameters(), self.student.parameters()):
            t_param.data = self.alpha * t_param.data + (1 - self.alpha) * s_param.data

print("GuidedBoxModel class defined!")


GuidedBoxModel class defined!


In [9]:
# Copy the IJmondBboxDataset class
class IJmondBboxDataset(Dataset):
    def __init__(self, records, img_npy_dir, img_size=256):
        self.records = records
        self.img_npy_dir = img_npy_dir
        self.img_size = img_size
        
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        record = self.records[idx]
        
        img_path = os.path.join(self.img_npy_dir, f"{record['id']}.npy")
        image = np.load(img_path)
        
        h_img, w_img = record['h_image'], record['w_image']
        x_bbox, y_bbox = record['x_bbox'], record['y_bbox']
        w_bbox, h_bbox = record['w_bbox'], record['h_bbox']
        
        actual_h, actual_w = image.shape[:2]
        scale_x = actual_w / w_img
        scale_y = actual_h / h_img
        
        x_bbox_scaled = int(x_bbox * scale_x)
        y_bbox_scaled = int(y_bbox * scale_y)
        w_bbox_scaled = int(w_bbox * scale_x)
        h_bbox_scaled = int(h_bbox * scale_y)
        
        pseudo_mask = np.zeros((actual_h, actual_w), dtype=np.float32)
        x1 = max(0, x_bbox_scaled)
        y1 = max(0, y_bbox_scaled)
        x2 = min(actual_w, x_bbox_scaled + w_bbox_scaled)
        y2 = min(actual_h, y_bbox_scaled + h_bbox_scaled)
        pseudo_mask[y1:y2, x1:x2] = 1.0
        
        image_pil = Image.fromarray(image).resize((self.img_size, self.img_size), Image.BILINEAR)
        mask_pil = Image.fromarray((pseudo_mask * 255).astype(np.uint8)).resize(
            (self.img_size, self.img_size), Image.NEAREST
        )
        
        image_tensor = TF.to_tensor(image_pil)
        mask_tensor = torch.from_numpy(np.array(mask_pil)).float() / 255.0
        mask_tensor = mask_tensor.unsqueeze(0)
        
        image_tensor = TF.normalize(image_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
        box_tensor = torch.tensor([
            x_bbox / w_img,
            y_bbox / h_img,
            w_bbox / w_img,
            h_bbox / h_img,
        ], dtype=torch.float32)
        
        return image_tensor, box_tensor, mask_tensor

print("IJmondBboxDataset class defined!")


IJmondBboxDataset class defined!


In [ ]:
# Define hyperparameters for models to load
PARAMS = {
    'learning_rate': 0.0001,
    'ema_alpha': 0.9,
    'img_size': 512,
    'positive_weights': [3, 5, 7, 9]
}

# Setup paths
# Update these paths based on where your grid_search results are located
GRID_SEARCH_DIR = Path('bbox_learn\bbox_learn\grid_search')  # Local download location
CONFIG_PATH = Path("bbox_learn/bbox_learn/guidedbox_config.yaml")  # Config file path
BBOX_LABELS_PATH = Path("bbox_learn/bbox_learn/dataset/ijmond_bbox/bbox_labels_1_aug_2025.json")
IMG_NPY_PATH = Path("bbox_learn/bbox_learn/dataset/ijmond_bbox/img_npy")

print(f"Grid search dir: {GRID_SEARCH_DIR}")
print(f"Config path: {CONFIG_PATH}")
print(f"Bbox labels: {BBOX_LABELS_PATH}")
print(f"Image NPY dir: {IMG_NPY_PATH}")

# Load config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cpu')  # or 'cuda' if you want GPU
print(f"\nUsing device: {device}")
print(f"Config loaded: num_classes={config['num_classes']}, batch_size={config['batch_size']}")


Grid search dir: grid_search
Config path: bbox_learn\bbox_learn\guidedbox_config.yaml
Bbox labels: bbox_learn\bbox_learn\dataset\ijmond_bbox\bbox_labels_1_aug_2025.json
Image NPY dir: bbox_learn\bbox_learn\dataset\ijmond_bbox\img_npy

Using device: cpu
Config loaded: num_classes=1, batch_size=8


In [16]:
# Load dataset and records
with open(BBOX_LABELS_PATH, 'r') as f:
    bbox_data = json.load(f)

all_records = bbox_data['data']
npy_ids = set(f.replace('.npy', '') for f in os.listdir(IMG_NPY_PATH) if f.endswith('.npy'))
matched_records = [r for r in all_records if str(r['id']) in npy_ids]

SMOKE_STATES = {3, 4, 9, 10, 11, 13, 15}
NO_SMOKE_STATES = {5, 12, 14}

smoke_records = [r for r in matched_records if r['label_state'] in SMOKE_STATES]
no_smoke_records = [r for r in matched_records if r['label_state'] in NO_SMOKE_STATES]
train_records = smoke_records + no_smoke_records

print(f"Total matched records: {len(matched_records)}")
print(f"Smoke records: {len(smoke_records)}")
print(f"No-smoke records: {len(no_smoke_records)}")
print(f"Training records: {len(train_records)}")

# Create dataset and split
full_dataset = IJmondBboxDataset(train_records, str(IMG_NPY_PATH), img_size=PARAMS['img_size'])
val_size = int(len(full_dataset) * 0.2)
train_size = len(full_dataset) - val_size
_, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

print(f"\nDataset created: img_size={PARAMS['img_size']}")
print(f"Validation set size: {len(val_dataset)}")


Total matched records: 610
Smoke records: 177
No-smoke records: 433
Training records: 610

Dataset created: img_size=512
Validation set size: 122


In [17]:
# Load all models for the specified hyperparameters
models_dict = {}

for pw in PARAMS['positive_weights']:
    model_name = f"lr{PARAMS['learning_rate']}_ema{PARAMS['ema_alpha']}_img{PARAMS['img_size']}_pw{pw}"
    checkpoint_path = GRID_SEARCH_DIR / f"{model_name}.pth"
    
    if not checkpoint_path.exists():
        print(f"WARNING: Model not found: {checkpoint_path}")
        continue
    
    # Create model
    run_config = config.copy()
    run_config['ema_alpha'] = PARAMS['ema_alpha']
    model = GuidedBoxModel(run_config).to(device)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    models_dict[pw] = {
        'model': model,
        'checkpoint_path': checkpoint_path,
        'best_iou': checkpoint.get('best_iou', 'N/A'),
    }
    
    print(f"✓ Loaded model (pw={pw}): {checkpoint_path.name}")
    print(f"  Best IoU: {checkpoint.get('best_iou', 'N/A'):.4f}")

print(f"\nTotal models loaded: {len(models_dict)}")



Total models loaded: 0


In [18]:
def visualize_batch_predictions(model, dataloader, num_samples=10, pw=None, title_prefix=""):
    """
    Visualize predictions for multiple samples.
    For each sample: raw image, prediction, ground truth, overlay
    """
    model.eval()
    
    # Denormalization constants
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    batch_idx = 0
    sample_count = 0
    
    with torch.no_grad():
        for images, boxes, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            # Get predictions
            outputs = model(images, positive_weight=pw)
            predictions = torch.sigmoid(outputs).cpu()
            
            # Process each sample in batch
            for i in range(len(images)):
                if sample_count >= num_samples:
                    return
                
                img = images[i].cpu() * std + mean
                img = img.permute(1, 2, 0).numpy().clip(0, 1)
                
                gt_mask = masks[i, 0].cpu().numpy()
                pred_mask = predictions[i, 0].numpy()
                pred_binary = (pred_mask > 0.5).astype(float)
                
                # Create figure with 4 subplots
                fig, axes = plt.subplots(1, 4, figsize=(16, 4))
                
                # Raw image
                axes[0].imshow(img)
                axes[0].set_title("Raw Image")
                axes[0].axis('off')
                
                # Ground truth mask
                axes[1].imshow(gt_mask, cmap='gray', vmin=0, vmax=1)
                axes[1].set_title("Ground Truth Mask")
                axes[1].axis('off')
                
                # Model prediction (probability)
                axes[2].imshow(pred_mask, cmap='hot', vmin=0, vmax=1)
                axes[2].set_title("Prediction (Probability)")
                axes[2].axis('off')
                
                # Overlay prediction on raw image
                axes[3].imshow(img)
                axes[3].imshow(pred_binary, alpha=0.5, cmap='Reds')
                axes[3].set_title("Overlay (Pred > 0.5)")
                axes[3].axis('off')
                
                plt.suptitle(f"{title_prefix} - Sample {sample_count+1}/{num_samples}", fontsize=12, fontweight='bold')
                plt.tight_layout()
                plt.show()
                
                sample_count += 1
                
            batch_idx += 1

print("Visualization function defined!")


Visualization function defined!


In [19]:
# Visualize predictions for each positive_weight model
for pw in sorted(PARAMS['positive_weights']):
    if pw not in models_dict:
        print(f"\nSkipping pw={pw} (model not loaded)")
        continue
    
    print(f"\n{'='*70}")
    print(f"Visualizing Predictions - positive_weight={pw}")
    print(f"{'='*70}")
    print(f"Best IoU from training: {models_dict[pw]['best_iou']:.4f}")
    
    model = models_dict[pw]['model']
    
    # Create validation dataloader
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    
    # Visualize 10 samples
    title = f"LR=0.0001, EMA=0.9, ImgSize=512, PW={pw}"
    visualize_batch_predictions(model, val_loader, num_samples=10, pw=pw, title_prefix=title)



Skipping pw=3 (model not loaded)

Skipping pw=5 (model not loaded)

Skipping pw=7 (model not loaded)

Skipping pw=9 (model not loaded)
